In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
STEP3.2 と STEP3.3 の入力データ（OH/FP）のサイズと変数差異を一括で比較。
- コンソールに要約を表示
- CSVで詳細（共有列／片側のみの列、数値列の共有など）を保存
- Jupyter でもそのまま実行可（argparse なし）

出力先:
  <base_dir>/diagnostics_inputs/compare_32_33_<timestamp>/<OH|FP>/ に CSV/JSON を保存
"""

import os, json, time
from pathlib import Path
import pandas as pd
import numpy as np

# =========================
# 設定（必要に応じて編集）
# =========================
BASE_DIR = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")

# 3.2 と 3.3 の入力ファイル対応（OH/FP）
FILES_32 = {
    "OH": BASE_DIR / "preprocessed_features_OH.csv",
    "FP": BASE_DIR / "preprocessed_features_FP.csv",
}
FILES_33 = {
    "OH": BASE_DIR / "DataMerge20211220oh_20250717_OH_delSMILES.csv",
    "FP": BASE_DIR / "DataMerge20211220FP_20250715.csv",
}

# 先頭列をIDとして扱うか（あなたのデータ前提）
USE_FIRST_COL_AS_INDEX = True

# 目的変数（必要なら除外判定に使用）
TARGETS = ["PCEmax", "Jsc", "Voc", "FF"]

# =========================
# ユーティリティ
# =========================
def ts():
    return time.strftime("%Y%m%d_%H%M%S")

def load_csv(path: Path, use_index=True) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"ファイルが見つかりません: {path}")
    df = pd.read_csv(path, header=0)
    if use_index:
        df = df.set_index(df.columns[0])
        df.index.name = "ID"
    return df

def split_numeric(df: pd.DataFrame):
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    non_cols = [c for c in df.columns if c not in num_cols]
    return num_cols, non_cols

def ensure_dir(*parts) -> Path:
    d = Path(*parts)
    d.mkdir(parents=True, exist_ok=True)
    return d

def save_series_as_csv(series, path: Path, name="column"):
    if isinstance(series, set):
        series = sorted(series)
    pd.Series(list(series)).to_csv(path, index=False, header=[name])

def compare_one(dataset_name: str, f32: Path, f33: Path, out_root: Path):
    print(f"\n=== [{dataset_name}] 比較開始 ===")
    print(f"[3.2] {f32}")
    print(f"[3.3] {f33}")

    out_ds = ensure_dir(out_root, dataset_name)

    # --- 読み込み
    A = load_csv(f32, use_index=USE_FIRST_COL_AS_INDEX)  # 3.2
    B = load_csv(f33, use_index=USE_FIRST_COL_AS_INDEX)  # 3.3

    # --- 形状
    print(f"[SIZE] 3.2: rows={A.shape[0]} cols={A.shape[1]} | 3.3: rows={B.shape[0]} cols={B.shape[1]}")

    # --- 全列スキーマ差分
    colsA, colsB = set(A.columns), set(B.columns)
    shared_all = colsA & colsB
    onlyA_all  = colsA - colsB
    onlyB_all  = colsB - colsA

    print(f"[SCHEMA(all)] 共有={len(shared_all)} | 3.2のみ={len(onlyA_all)} | 3.3のみ={len(onlyB_all)}")

    # --- 数値列のみでの比較
    numA, _ = split_numeric(A)
    numB, _ = split_numeric(B)
    shared_num = set(numA) & set(numB)
    onlyA_num  = set(numA) - set(numB)
    onlyB_num  = set(numB) - set(numA)
    print(f"[SCHEMA(numeric)] 共有数値列={len(shared_num)} | 3.2のみ数値={len(onlyA_num)} | 3.3のみ数値={len(onlyB_num)}")

    # --- 目的変数の有無（参考）
    present_targets_A = [t for t in TARGETS if t in A.columns]
    present_targets_B = [t for t in TARGETS if t in B.columns]
    print(f"[TARGETS] 3.2に含まれる: {present_targets_A} | 3.3に含まれる: {present_targets_B}")

    # --- 欠損率（共有列のみ抜粋・上位10差分表示用）
    missA = A.isna().mean()
    missB = B.isna().mean()
    miss_df = pd.DataFrame({"missing_32": missA, "missing_33": missB})
    miss_df["delta_32_minus_33"] = miss_df["missing_32"] - miss_df["missing_33"]
    miss_shared = miss_df.loc[sorted(shared_all)]
    top_miss = miss_shared["delta_32_minus_33"].abs().sort_values(ascending=False).head(10)
    print("\n[欠損率 |Δ(3.2-3.3)| 上位10（共有列）]")
    print(top_miss)

    # --- 出力保存
    # 一覧CSV
    save_series_as_csv(sorted(shared_all), out_ds / "shared_columns_all.csv", name="column")
    save_series_as_csv(sorted(onlyA_all),  out_ds / "only_in_3.2_all.csv", name="column")
    save_series_as_csv(sorted(onlyB_all),  out_ds / "only_in_3.3_all.csv", name="column")
    save_series_as_csv(sorted(shared_num), out_ds / "shared_columns_numeric.csv", name="column")
    save_series_as_csv(sorted(onlyA_num),  out_ds / "only_in_3.2_numeric.csv", name="column")
    save_series_as_csv(sorted(onlyB_num),  out_ds / "only_in_3.3_numeric.csv", name="column")
    miss_df.to_csv(out_ds / "missingness_all_columns.csv")

    # サマリーJSON
    summary = {
        "dataset": dataset_name,
        "file_3_2": str(f32),
        "file_3_3": str(f33),
        "size_3_2": {"rows": int(A.shape[0]), "cols": int(A.shape[1])},
        "size_3_3": {"rows": int(B.shape[0]), "cols": int(B.shape[1])},
        "shared_all_columns": len(shared_all),
        "only_in_3_2_all": len(onlyA_all),
        "only_in_3_3_all": len(onlyB_all),
        "shared_numeric_columns": len(shared_num),
        "only_in_3_2_numeric": len(onlyA_num),
        "only_in_3_3_numeric": len(onlyB_num),
        "targets_in_3_2": present_targets_A,
        "targets_in_3_3": present_targets_B,
    }
    with open(out_ds / "SUMMARY.json", "w") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print(f"\n[保存] {out_ds} に CSV/JSON を出力しました。")
    print(f"=== [{dataset_name}] 完了 ===")

def main():
    out_root = ensure_dir(BASE_DIR, "diagnostics_inputs", f"compare_32_33_{ts()}")
    # OH / FP を一括処理
    for ds in ["OH", "FP"]:
        try:
            compare_one(ds, FILES_32[ds], FILES_33[ds], out_root)
        except Exception as e:
            print(f"\n[ERROR][{ds}] {e}")

    print(f"\n✅ 全データセット完了 → {out_root}")

if __name__ == "__main__":
    main()


=== [OH] 比較開始 ===
[3.2] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/preprocessed_features_OH.csv
[3.3] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/DataMerge20211220oh_20250717_OH_delSMILES.csv
[SIZE] 3.2: rows=1046 cols=394 | 3.3: rows=1046 cols=412
[SCHEMA(all)] 共有=394 | 3.2のみ=0 | 3.3のみ=18
[SCHEMA(numeric)] 共有数値列=394 | 3.2のみ数値=0 | 3.3のみ数値=18
[TARGETS] 3.2に含まれる: [] | 3.3に含まれる: ['PCEmax', 'Jsc', 'Voc', 'FF']

[欠損率 |Δ(3.2-3.3)| 上位10（共有列）]
ALthickness                        0.0
Additivename_1..8.Diiodooctane.    0.0
Additivename_1.8dibromoooctane     0.0
Additivename_1.8dicholorooctane    0.0
Additivename_1.8dicyanooctane      0.0
Additivename_1.8diiodooctane       0.0
Additivename_1.8octanediacetate    0.0
Additivename_1.8octanedithiol      0.0
Additivename_1chloronaphthalene    0.0
Additivename_diphenylether         0.0
Name: delta_32_minus_33, dtype: float64

[保存] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/diagnostics_inputs/compare_32_